In [3]:
!pip install -q chromadb langdetect sentence-transformers spacy pandas pyarrow pytest datasets

In [4]:
import os
!mkdir -p /content/internship/{scripts,src,tests,data/raw,data/processed}
os.chdir("/content/internship")
!git init -q
!python -m venv venv --without-pip
!touch src/__init__.py

with open(".gitignore", "w") as f:
    f.write("data/\n.env\nvenv/\n__pycache__/\n*.pyc\n")
!cat .gitignore

data/
.env
venv/
__pycache__/
*.pyc


In [5]:
from importlib.metadata import version
pkgs = ["pandas", "sentence-transformers", "chromadb", "spacy", "langdetect", "pytest", "datasets", "pyarrow", "torch"]
with open("requirements.txt", "w") as f:
    for p in pkgs:
        f.write(f"{p}=={version(p)}\n")
!cat requirements.txt

pandas==2.2.3
sentence-transformers==5.7.0
chromadb==1.5.9
spacy==3.8.18
langdetect==1.0.9
pytest==8.4.2
datasets==4.8.5
pyarrow==23.0.1
torch==2.14.1+cu130


In [6]:
%%writefile scripts/verify_env.py
import torch

assert torch.cuda.is_available(), "GPU/CUDA not available!"
print("GPU:", torch.cuda.get_device_name(0))

import pandas, sentence_transformers, chromadb, spacy
print("All imports OK: pandas, sentence-transformers, chromadb, spacy")

Writing scripts/verify_env.py


In [7]:
!python scripts/verify_env.py

GPU: Tesla T4
All imports OK: pandas, sentence-transformers, chromadb, spacy


In [8]:
import json
from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train", streaming=True)

count = 0
with open("data/raw/wiki_raw.jsonl", "w") as f:
    for row in ds:
        f.write(json.dumps({"id": row["id"], "text": row["text"]}) + "\n")
        count += 1
        if count >= 6000:
            break
print("Saved documents:", count)

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Saved documents: 6000


In [9]:
%%writefile src/preprocessing.py
import re, html, unicodedata
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

def strip_html(text):
    text = re.sub(r"<[^>]+>", " ", text)   # tags hatao
    return html.unescape(text)             # &amp; -> &

def normalize_unicode(text):
    return unicodedata.normalize("NFKC", text)

def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def is_english(text):
    try:
        return detect(text[:1000]) == "en"
    except Exception:
        return False

def clean_text(text):
    return clean_whitespace(normalize_unicode(strip_html(text)))

Writing src/preprocessing.py


In [11]:
%%writefile tests/test_preprocessing.py
from src.preprocessing import strip_html, normalize_unicode, clean_whitespace, is_english, clean_text

def test_strip_html():
    assert strip_html("<p>Hello &amp; welcome</p>").strip() == "Hello & welcome"

def test_normalize_unicode():
    assert normalize_unicode("ｈｅｌｌｏ") == "hello"

def test_clean_whitespace():
    assert clean_whitespace("  a   b \n\n c  ") == "a b c"

def test_is_english():
    assert is_english("This is a simple English sentence for testing.")
    assert not is_english("Ceci est une phrase écrite en français pour le test.")

def test_clean_text():
    assert clean_text("<b>Hi</b>   there") == "Hi there"

Overwriting tests/test_preprocessing.py


In [12]:
!python -m pytest tests -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/internship
plugins: langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0, platformdirs-4.14.3
collected 5 items                                                              

tests/test_preprocessing.py::test_strip_html PASSED                      [ 20%]
tests/test_preprocessing.py::test_normalize_unicode PASSED               [ 40%]
tests/test_preprocessing.py::test_clean_whitespace PASSED                [ 60%]
tests/test_preprocessing.py::test_is_english PASSED                      [ 80%]
tests/test_preprocessing.py::test_clean_text PASSED                      [100%]

============================== 5 passed in 0.32s ===============================


In [17]:
from src.preprocessing import strip_html, normalize_unicode, clean_whitespace, clean_text, is_english

# 1) HTML entities + links + bohat saari new lines
dirty2 = "<div class='box'>  <p>Python&nbsp;is   great!&nbsp;&lt;3</p>\n\n\n<a href='x.com'>Click   here</a> &copy; 2024 </div>"

# 2) Unicode normalization (ligature, circled numbers)
uni = "ﬁnal ① ② ③ ｔｅｓｔ"

# 3) Alag languages
english = "Python is a popular programming language used for data science."
spanish = "El aprendizaje automático es una rama de la inteligencia artificial."
german  = "Maschinelles Lernen ist ein Teilgebiet der künstlichen Intelligenz."

print("Step by step:")
print("strip_html       :", repr(strip_html(dirty2)))
print("clean_text       :", repr(clean_text(dirty2)))
print()
print("Unicode BEFORE   :", uni)
print("Unicode AFTER    :", normalize_unicode(uni))
print()
print("English ->", is_english(english))   # True
print("Spanish ->", is_english(spanish))   # False
print("German  ->", is_english(german))    # False

Step by step:
strip_html       : '    Python\xa0is   great!\xa0<3 \n\n\n Click   here  © 2024  '
clean_text       : 'Python is great! <3 Click here © 2024'

Unicode BEFORE   : ﬁnal ① ② ③ ｔｅｓｔ
Unicode AFTER    : final 1 2 3 test

English -> True
Spanish -> False
German  -> False


In [18]:
import pandas as pd
from src.preprocessing import clean_text, is_english

df = pd.read_json("data/raw/wiki_raw.jsonl", lines=True)
df["text"] = df["text"].apply(clean_text)
df = df[df["text"].str.len() > 200]                 # bohat chhote docs hatao
df = df[df["text"].apply(is_english)]               # sirf English
df = df.drop_duplicates(subset="text").reset_index(drop=True)

# Validation
assert len(df) >= 5000, f"Only {len(df)} docs, need 5000+"
assert df["text"].notna().all()

df.to_parquet("data/processed/clean_corpus.parquet", index=False)
print("Clean docs:", len(df))
df.head()

Clean docs: 5732


,id,text
0,1,April (Apr.) is the fourth month of the year i...
1,2,August (Aug.) is the eighth month of the year ...
2,6,Art is a creative activity. It produces a prod...
3,8,A or a is the first letter of the English alph...
4,9,Air is the Earth's atmosphere. Air is a mixtur...


In [19]:
!git config user.email "you@example.com"
!git config user.name "Your Name"
!git add .
!git commit -q -m "Week 1: env setup, data ingestion, preprocessing, tests"
!git status --short; git log --oneline

On branch master
nothing to commit, working tree clean
5e4cfe4 (HEAD -> master) Week 1: env setup, data ingestion, preprocessing, tests
